<a href="https://colab.research.google.com/github/TesterSim2/Focused-IDE/blob/main/Closing_Loop_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: THE SEMANTIC SUBSTRATE FOUNDATION
# ==========================================
!pip install -q "tree-sitter>=0.23.0" "tree-sitter-python>=0.23.0"

import hashlib
import logging
from dataclasses import dataclass, field
from typing import List, Tuple

# 1. Configure robust logging for the Colab Environment
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger("AST_Engine")

# 2. Content-Aware Semantic Hashing (CASH)
def generate_cash_uuid(structural_path: str, node_type: str, node_text: str) -> str:
    """
    Generates a deterministic Content-Aware Semantic Hash (CASH) for an AST node.
    This algorithm guarantees UUID stability across sibling insertions/deletions,
    preventing the 'Sibling Index Shift' vulnerability during Rebase operations.

    Args:
        structural_path (str): The lineage of the node (e.g., 'Module > FunctionDef > IfStatement').
        node_type (str): The native Tree-sitter node type.
        node_text (str): The raw source code of the node.

    Returns:
        str: An 8-character uppercase UUID mapped as XXXX-XXXX.
    """
    # Extract the localized signature (the first non-empty line of the node's text).
    # This prevents the hash from changing if a deep, hidden child node is modified.
    lines =[line.strip() for line in node_text.splitlines() if line.strip()]
    localized_signature = lines[0] if lines else ""

    # Create a micro-hash of the localized signature
    sig_hash = hashlib.sha256(localized_signature.encode('utf-8')).hexdigest()[:8]

    # Combine structural path, node type, and signature hash to create the deterministic seed
    deterministic_string = f"{structural_path}::{node_type}::{sig_hash}"

    # Generate the final UUID
    final_hash = hashlib.sha256(deterministic_string.encode('utf-8')).hexdigest()

    # Format for maximal readability by the LLM tokenizer (e.g., A1B2-C3D4)
    return f"{final_hash[:4].upper()}-{final_hash[4:8].upper()}"

# 3. The Immutable AST Data Structure
@dataclass
class ASTNode:
    """
    The immutable, Virtual DOM representation of a Tree-sitter AST node.
    This acts as the single source of truth for our MCTS State Manager.
    """
    uuid: str
    node_type: str
    structural_path: str
    start_byte: int
    end_byte: int
    start_point: Tuple[int, int]  # (row, column)
    end_point: Tuple[int, int]    # (row, column)
    text: str                     # The raw source code of this node
    children: List['ASTNode'] = field(default_factory=list)
    is_folded: bool = False       # Tracks the Semantic Folding state for the UI

    def __repr__(self) -> str:
        return f"<ASTNode[{self.uuid}] {self.node_type} (Lines: {self.start_point[0]}-{self.end_point[0]})>"

logger.info("Cell 1 Execution Complete: Modern Tree-sitter, CASH UUID generator, and ASTNode dataclass initialized.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.4/635.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 13.4 MB/s eta 0:00:00


In [ ]:
# ==========================================
# CELL 2: THE SEMANTIC GRAPH MANAGER
# ==========================================
import tree_sitter
import tree_sitter_python
from typing import Optional

print("Initializing Tree-sitter Python parser...")

# Modern Tree-sitter (>=0.23.0) API initialization
try:
    PY_LANGUAGE = tree_sitter.Language(tree_sitter_python.language())
    parser = tree_sitter.Parser(PY_LANGUAGE)
except Exception as e:
    # Defensive fallback for edge-case environment variations
    print(f"Applying API fallback: {e}")
    PY_LANGUAGE = tree_sitter.Language(tree_sitter_python.language(), "python")
    parser = tree_sitter.Parser()
    parser.set_language(PY_LANGUAGE)

class SemanticGraphManager:
    """
    Ingests source code, parses it via Tree-sitter, and recursively builds
    our immutable DAG of ASTNodes with deterministic CASH UUIDs.
    """
    def __init__(self, parser: tree_sitter.Parser):
        self.parser = parser

    def parse_to_graph(self, source_code: str, file_path: str = "root") -> ASTNode:
        """
        Converts raw Python source code into our Virtual DOM representation.
        """
        source_bytes = source_code.encode('utf-8')
        tree = self.parser.parse(source_bytes)

        # The root node is the absolute top of the module
        root_ts_node = tree.root_node

        return self._build_node(
            ts_node=root_ts_node,
            source_bytes=source_bytes,
            structural_path=file_path
        )

    def _build_node(self, ts_node, source_bytes: bytes, structural_path: str) -> Optional[ASTNode]:
        """
        Recursively traverses the tree-sitter graph.
        We only capture 'named' nodes (which have semantic meaning) to skip
        structural punctuation (like standalone brackets) and save LLM tokens.
        """
        if not ts_node.is_named:
            return None

        # Extract the raw text of this specific node
        node_text = source_bytes[ts_node.start_byte:ts_node.end_byte].decode('utf-8')

        # Update the lineage path to guarantee UUID determinism across the tree
        current_path = f"{structural_path} > {ts_node.type}"

        # Generate our deterministic CASH UUID
        uuid = generate_cash_uuid(
            structural_path=current_path,
            node_type=ts_node.type,
            node_text=node_text
        )

        # Initialize our immutable DOM node
        ast_node = ASTNode(
            uuid=uuid,
            node_type=ts_node.type,
            structural_path=current_path,
            start_byte=ts_node.start_byte,
            end_byte=ts_node.end_byte,
            start_point=ts_node.start_point,
            end_point=ts_node.end_point,
            text=node_text,
            children=[],
            is_folded=False
        )

        # Recursively build children
        for child in ts_node.children:
            child_ast_node = self._build_node(child, source_bytes, current_path)
            if child_ast_node is not None:
                ast_node.children.append(child_ast_node)

        return ast_node

# Instantiate the manager
graph_manager = SemanticGraphManager(parser)

# ==========================================
# DIAGNOSTIC TEST: Prove the Substrate Works
# ==========================================
test_code = """
class DataProcessor:
    def __init__(self):
        self.data =[]

    def process(self):
        if True:
            print("Running")
"""

print("Parsing test code into AST Graph...")
root_node = graph_manager.parse_to_graph(test_code, file_path="test_module.py")

# A quick recursive printer to visualize the DAG structure
def print_tree(node: ASTNode, depth: int = 0):
    indent = "  " * depth
    # Only print block-level nodes to keep the visual clean
    if node.node_type in ['module', 'class_definition', 'function_definition', 'if_statement', 'expression_statement']:
        print(f"{indent}# @node: {node.uuid} [{node.node_type}]")
    for child in node.children:
        print_tree(child, depth + 1)

print("\n--- IMMUTABLE DAG STRUCTURE ---")
print_tree(root_node)
print("-------------------------------")
print("Cell 2 Execution Complete: Semantic Graph Manager is operational.")

Initializing Tree-sitter Python parser...
Parsing test code into AST Graph...

--- IMMUTABLE DAG STRUCTURE ---
# @node: 64F9-25AA [module]
  # @node: C737-ED8E [class_definition]
      # @node: 32B3-BA85 [function_definition]
          # @node: 86B5-2FA2 [expression_statement]
      # @node: 89E8-D339 [function_definition]
          # @node: 2CEE-7AA4 [if_statement]
              # @node: 95E9-81BF [expression_statement]
-------------------------------
Cell 2 Execution Complete: Semantic Graph Manager is operational.


In [ ]:
# ==========================================
# CELL 3: THE VIRTUAL DOM RENDERER (ZONE 1)
# ==========================================
from typing import Dict, List, Tuple

class VirtualDOMRenderer:
    """
    Renders the Immutable AST Graph into the 'Zone 1' UI for the LLM Context Window.
    Uses an Intelligent Overlay technique to preserve all original formatting while
    injecting language-native BPE anchors (# @node: UUID) and semantic folds.
    """
    def __init__(self, source_code: str):
        self.lines = source_code.splitlines()

    def render(self, root_node: ASTNode) -> str:
        tags_by_line: Dict[int, ASTNode] = {}
        folds: List[Tuple[int, int]] =[]

        # We only want to tag structural nodes to keep the Context Window clean.
        # We omit tiny inline variables to prevent token bloat.
        TRACKED_TYPES = {
            'class_definition', 'function_definition',
            'if_statement', 'for_statement', 'while_statement',
            'try_statement', 'with_statement', 'expression_statement',
            'return_statement'
        }

        def traverse(node: ASTNode):
            if node.node_type in TRACKED_TYPES:
                line_idx = node.start_point[0]
                # If multiple nodes start on the same line (e.g., if True: break)
                # we keep the outermost node (largest byte span) to avoid double-tagging.
                if line_idx not in tags_by_line:
                    tags_by_line[line_idx] = node
                else:
                    existing = tags_by_line[line_idx]
                    if (node.end_byte - node.start_byte) > (existing.end_byte - existing.start_byte):
                        tags_by_line[line_idx] = node

            # If folded, register the fold range and STOP traversing children.
            if node.is_folded:
                # Fold from the line immediately after the signature, down to the end of the block.
                folds.append((node.start_point[0] + 1, node.end_point[0]))
                return

            for child in node.children:
                traverse(child)

        # 1. Map the AST constraints
        traverse(root_node)

        output: List[str] =[]
        skip_until = -1

        # 2. Render the overlay
        for i, line in enumerate(self.lines):
            if i < skip_until:
                continue

            # A. Inject the UUID tag if a tracked node starts on this line
            node = tags_by_line.get(i)
            if node:
                indent = len(line) - len(line.lstrip())
                indent_str = " " * indent
                output.append(f"{indent_str}# @node: {node.uuid}")

            # B. Output the actual source code line
            output.append(line)

            # C. Handle Semantic Folding execution
            for f_start, f_end in folds:
                if i == f_start - 1 and f_end >= f_start:
                    indent = len(line) - len(line.lstrip())
                    # Indent the placeholder deeper than the signature for standard Python visuals
                    indent_str = " " * (indent + 4)
                    hidden_count = (f_end - f_start) + 1
                    output.append(f"{indent_str}... (Hidden: {hidden_count} lines)")
                    skip_until = f_end + 1
                    break

        return "\n".join(output)

# ==========================================
# DIAGNOSTIC TEST: Semantic Folding UI
# ==========================================
expanded_test_code = """class AdvancedProcessor:
    def __init__(self, data):
        self.data = data
        self.is_ready = False
        self.logger = None

    def normalize(self):
        for item in self.data:
            if item < 0:
                print("Warning: Negative value")
                continue
            print(f"Processing {item}")
"""

print("Building AST Graph for AdvancedProcessor...")
adv_root = graph_manager.parse_to_graph(expanded_test_code, file_path="advanced.py")

# Helper function to find a node by name for our test
def set_fold_by_name(node: ASTNode, name: str):
    if name in node.text and node.node_type == 'function_definition':
        node.is_folded = True
    for child in node.children:
        set_fold_by_name(child, name)

# Let's forcefully fold the `__init__` function to prove the UI works
set_fold_by_name(adv_root, "__init__")

print("\nRendering Zone 1 UI (With __init__ folded)...\n")
renderer = VirtualDOMRenderer(expanded_test_code)
zone_1_ui = renderer.render(adv_root)

print("=== [ZONE 1: CURRENT FILE STATE] ===")
print(zone_1_ui)
print("====================================")
print("\nCell 3 Execution Complete: Virtual DOM Renderer is operational.")

Building AST Graph for AdvancedProcessor...

Rendering Zone 1 UI (With __init__ folded)...

=== [ZONE 1: CURRENT FILE STATE] ===
# @node: B711-A769
class AdvancedProcessor:
    # @node: C920-EB42
    def __init__(self, data):
        ... (Hidden: 3 lines)
        
    # @node: 063B-7DEB
    def normalize(self):
        # @node: 4313-79A9
        for item in self.data:
            # @node: EFDF-AD5D
            if item < 0:
                # @node: 821C-B048
                print("Warning: Negative value")
                continue
            # @node: AE4B-DD07
            print(f"Processing {item}")

Cell 3 Execution Complete: Virtual DOM Renderer is operational.


In [ ]:
# ==========================================
# CELL 4: THE ACTUATION QUARANTINE SANDBOX
# ==========================================
class ActuationError(Exception):
    """Raised when an LLM actuation violates structural or syntactic invariants."""
    pass

class GraphMutator:
    """
    Executes AST mutations securely. Enforces Contextual Completeness
    and utilizes Tree-sitter as a strict syntax firewall.
    """
    def __init__(self, graph_manager: SemanticGraphManager):
        self.graph_manager = graph_manager

    def _find_node(self, root: ASTNode, target_uuid: str) -> Optional[ASTNode]:
        """Depth-first search to locate an ASTNode by its deterministic UUID."""
        if root.uuid == target_uuid:
            return root
        for child in root.children:
            res = self._find_node(child, target_uuid)
            if res:
                return res
        return None

    def _has_folded_descendants(self, node: ASTNode) -> bool:
        """Checks if a node or any of its children are currently folded in the UI."""
        if node.is_folded:
            return True
        return any(self._has_folded_descendants(c) for c in node.children)

    def mutate(self, root: ASTNode, current_source: str, action: str, target_uuid: str, new_content: str) -> Tuple[ASTNode, str]:
        """
        Attempts to apply a mutation. Returns a tuple of (New_AST_Root, New_Source_Code)
        if successful. Raises ActuationError if invalid.
        """
        target_node = self._find_node(root, target_uuid)
        if not target_node:
            raise ActuationError(f"UUID [{target_uuid}] not found in current AST.")

        # INVARIANT 1: The Contextual Completeness Check
        if action == "Replace_Node" and self._has_folded_descendants(target_node):
            raise ActuationError(
                f"Actuation Denied: Target node [{target_uuid}] contains unexpanded child nodes. "
                "You must Expand_Node before replacing to prevent data loss."
            )

        # Work entirely in bytes to guarantee accurate slicing
        # (especially with multibyte utf-8 characters)
        source_bytes = current_source.encode('utf-8')
        start_byte = target_node.start_byte
        end_byte = target_node.end_byte
        new_bytes = new_content.encode('utf-8')

        # Execute Graph Operation
        if action == "Replace_Node":
            new_source_bytes = source_bytes[:start_byte] + new_bytes + source_bytes[end_byte:]

        elif action == "Insert_Before_Node":
            # Match the indentation of the target node exactly
            indent = b" " * target_node.start_point[1]
            new_source_bytes = source_bytes[:start_byte] + new_bytes + b"\n" + indent + source_bytes[start_byte:]

        elif action == "Insert_After_Node":
            indent = b" " * target_node.start_point[1]
            new_source_bytes = source_bytes[:end_byte] + b"\n" + indent + new_bytes + source_bytes[end_byte:]

        else:
            raise ActuationError(f"Unknown action: {action}")

        # INVARIANT 2: The Syntax Quarantine
        tree = self.graph_manager.parser.parse(new_source_bytes)
        if tree.root_node.has_error:
            raise ActuationError(
                "Actuation Failed: The generated code introduces a structural syntax error. "
                "Check your indentation, unclosed brackets, or invalid statements."
            )

        # If we pass all invariants, decode and rebuild the MCTS state
        new_source_str = new_source_bytes.decode('utf-8')
        new_root = self.graph_manager.parse_to_graph(new_source_str)
        return new_root, new_source_str


# ==========================================
# DIAGNOSTIC TEST: Stress-Testing the Sandbox
# ==========================================
mutator = GraphMutator(graph_manager)
print("Initializing Graph Mutator Sandbox Tests...\n")

# We must extract the dynamic UUIDs generated in Cell 3 for testing.
# Let's find the UUID for the folded __init__ and the 'if item < 0:' statement.
init_uuid = None
if_uuid = None

def find_test_uuids(node):
    global init_uuid, if_uuid
    if "def __init__" in node.text: init_uuid = node.uuid
    if "if item < 0:" in node.text: if_uuid = node.uuid
    for c in node.children: find_test_uuids(c)

find_test_uuids(adv_root)

# TEST 1: The Blind Rewrite Vulnerability (Should Fail)
print(f"Test 1: Attempting Replace_Node on folded __init__ [UUID: {init_uuid}]")
try:
    mutator.mutate(adv_root, expanded_test_code, "Replace_Node", init_uuid, "def __init__(self):\n    pass")
    print("❌ FAIL: Sandbox allowed a blind rewrite!")
except ActuationError as e:
    print(f"✅ PASS: Sandbox intercepted. Reason: {e}")

# TEST 2: Hallucinated Syntax Injection (Should Fail)
print(f"\nTest 2: Attempting to insert broken syntax before 'if item < 0:' [UUID: {if_uuid}]")
broken_syntax = "for x in range(10) # Missing colon"
try:
    mutator.mutate(adv_root, expanded_test_code, "Insert_Before_Node", if_uuid, broken_syntax)
    print("❌ FAIL: Sandbox allowed broken syntax!")
except ActuationError as e:
    print(f"✅ PASS: Sandbox intercepted. Reason: {e}")

# TEST 3: Successful Graft (Should Pass)
print(f"\nTest 3: Attempting valid Insert_Before_Node at [UUID: {if_uuid}]")
valid_syntax = "print('Checking item bounds')"
try:
    new_root, new_source = mutator.mutate(adv_root, expanded_test_code, "Insert_Before_Node", if_uuid, valid_syntax)
    print("✅ PASS: Graft successful. Tree-sitter verified the syntax.")
    print("\n--- NEW ZONE 1 PREVIEW ---")
    print(renderer.render(new_root))
    print("--------------------------")
except ActuationError as e:
    print(f"❌ FAIL: Valid graft was rejected! {e}")

print("\nCell 4 Execution Complete: Actuation Quarantine Sandbox is operational.")

Initializing Graph Mutator Sandbox Tests...

Test 1: Attempting Replace_Node on folded __init__ [UUID: C920-EB42]
✅ PASS: Sandbox intercepted. Reason: Actuation Denied: Target node [C920-EB42] contains unexpanded child nodes. You must Expand_Node before replacing to prevent data loss.

Test 2: Attempting to insert broken syntax before 'if item < 0:' [UUID: EFDF-AD5D]
✅ PASS: Sandbox intercepted. Reason: Actuation Failed: The generated code introduces a structural syntax error. Check your indentation, unclosed brackets, or invalid statements.

Test 3: Attempting valid Insert_Before_Node at [UUID: EFDF-AD5D]
✅ PASS: Graft successful. Tree-sitter verified the syntax.

--- NEW ZONE 1 PREVIEW ---
# @node: A26A-D6CD
class AdvancedProcessor:
    # @node: 4717-A8D9
    def __init__(self, data):
        # @node: 4975-7593
        self.data = data
        # @node: DD65-A48F
        self.is_ready = False
        # @node: 9F51-AB5C
        self.logger = None
        
    # @node: 7EAC-96F1
    def

In [ ]:
# ==========================================
# CELL 5: THE MCTS VIRTUAL DOM DAEMON
# ==========================================
import copy

@dataclass
class StateCommit:
    """Represents a single, immutable point in the MCTS timeline."""
    turn_id: int
    file_path: str
    source_code: str
    ast_root: ASTNode
    action_log: str

class MCTSController:
    """
    Manages the timeline, handles state-transfer between AST generations,
    and guarantees O(1) Rollback capability via KV Cache alignment.
    """
    def __init__(self, initial_source: str, file_path: str, graph_manager: SemanticGraphManager):
        self.graph_manager = graph_manager
        self.mutator = GraphMutator(graph_manager)

        # Initialize Timeline
        self.history: List[StateCommit] =[]

        # Create Genesis State
        initial_ast = self.graph_manager.parse_to_graph(initial_source, file_path=file_path)
        self._commit(0, file_path, initial_source, initial_ast, "System Initialization")

    def get_current_state(self) -> StateCommit:
        return self.history[-1]

    def _commit(self, turn_id: int, file_path: str, source: str, ast: ASTNode, log: str):
        commit = StateCommit(turn_id, file_path, source, ast, log)
        self.history.append(commit)

    def _transfer_fold_state(self, old_root: ASTNode, new_root: ASTNode):
        """
        Transfers the UI fold state across the mutation boundary.
        Because CASH UUIDs are structurally immune to sibling insertions,
        we can confidently map fold states 1:1.
        """
        folded_uuids = set()
        def collect_folds(node: ASTNode):
            if node.is_folded:
                folded_uuids.add(node.uuid)
            for c in node.children:
                collect_folds(c)

        def apply_folds(node: ASTNode):
            if node.uuid in folded_uuids:
                node.is_folded = True
            for c in node.children:
                apply_folds(c)

        collect_folds(old_root)
        apply_folds(new_root)

    def apply_action(self, action: str, target_uuid: str, new_content: str) -> bool:
        """Attempts to execute a mutation and advance the MCTS timeline."""
        current = self.get_current_state()
        try:
            # 1. Mutate graph (passing the exact file_path to preserve CASH UUIDs)
            new_root, new_source = self.mutator.mutate(
                root=current.ast_root,
                current_source=current.source_code,
                action=action,
                target_uuid=target_uuid,
                new_content=new_content
            )

            # Re-parse with exact lineage to prevent UUID drift
            new_root = self.graph_manager.parse_to_graph(new_source, file_path=current.file_path)

            # 2. Transfer the UI state (Amnesia Fix)
            self._transfer_fold_state(current.ast_root, new_root)

            # 3. Commit to timeline
            log = f"Executed `{action}` on [UUID: {target_uuid}]"
            self._commit(current.turn_id + 1, current.file_path, new_source, new_root, log)
            return True

        except ActuationError as e:
            print(f"[SYSTEM INTERCEPT]: {e}")
            return False

    def rollback(self):
        """Executes O(1) KV-Cache optimized rollback."""
        if len(self.history) > 1:
            popped = self.history.pop()
            print(f"Rolled back Turn {popped.turn_id}. Reverted to Turn {self.get_current_state().turn_id}.")
        else:
            print("Already at Genesis state.")

    def render_zone_1(self) -> str:
        """Renders the current Virtual DOM (Phantom Code Fix)"""
        current = self.get_current_state()
        renderer = VirtualDOMRenderer(current.source_code)
        return renderer.render(current.ast_root)

# ==========================================
# DIAGNOSTIC TEST: The Timeline Continuity
# ==========================================
print("Initializing MCTS State Daemon...")

# 1. Start with the Genesis state
daemon = MCTSController(expanded_test_code, "advanced.py", graph_manager)
genesis_root = daemon.get_current_state().ast_root

# Robust recursive helper to find exact UUIDs safely
def get_node_by_type_and_text(node: ASTNode, n_type: str, text_snippet: str) -> Optional[ASTNode]:
    if node.node_type == n_type and text_snippet in node.text:
        return node
    for c in node.children:
        res = get_node_by_type_and_text(c, n_type, text_snippet)
        if res: return res
    return None

# 2. Artificially fold __init__ to prove State Transfer across mutations
init_node = get_node_by_type_and_text(genesis_root, 'function_definition', 'def __init__')
if init_node:
    init_node.is_folded = True
    print(f"Genesis State: Folded __init__[UUID: {init_node.uuid}]")

# 3. Find the exact 'if' statement
if_node = get_node_by_type_and_text(genesis_root, 'if_statement', 'if item < 0:')

if if_node:
    # 4. Apply the Mutation
    print(f"\nExecuting Mutation at Turn 1 on [UUID: {if_node.uuid}]...")
    success = daemon.apply_action("Insert_Before_Node", if_node.uuid, "print('Checking item bounds')")

    if success:
        print("\n=== [TURN 1 STATE] ===")
        print(daemon.render_zone_1())

        # 5. Prove O(1) Rollback
        print("\nExecuting MCTS Rollback...")
        daemon.rollback()

        print("\n===[TURN 0 STATE (RESTORED)] ===")
        print(daemon.render_zone_1())
        print("=================================")
else:
    print("Could not locate if_node for testing.")

print("\nCell 5 Execution Complete: MCTS Daemon operational.")

Initializing MCTS State Daemon...
Genesis State: Folded __init__[UUID: C920-EB42]

Executing Mutation at Turn 1 on [UUID: EFDF-AD5D]...

=== [TURN 1 STATE] ===
# @node: B711-A769
class AdvancedProcessor:
    # @node: C920-EB42
    def __init__(self, data):
        ... (Hidden: 3 lines)
        
    # @node: 063B-7DEB
    def normalize(self):
        # @node: 4313-79A9
        for item in self.data:
            # @node: D3EA-FE02
            print('Checking item bounds')
            # @node: EFDF-AD5D
            if item < 0:
                # @node: 821C-B048
                print("Warning: Negative value")
                continue
            # @node: AE4B-DD07
            print(f"Processing {item}")

Executing MCTS Rollback...
Rolled back Turn 1. Reverted to Turn 0.

===[TURN 0 STATE (RESTORED)] ===
# @node: B711-A769
class AdvancedProcessor:
    # @node: C920-EB42
    def __init__(self, data):
        ... (Hidden: 3 lines)
        
    # @node: 063B-7DEB
    def normalize(self):
   

In [ ]:
# ==========================================
# CELL 6: THE TELEMETRY ENGINE (ZONE 2)
# ==========================================
!pip install -q pyflakes

import ast
from pyflakes import api, reporter
from io import StringIO

class TelemetryEngine:
    """
    Acts as the Headless LSP. Analyzes the current virtual source code
    and returns deterministic semantic diagnostics (Zone 2).
    """
    def __init__(self):
        pass

    def get_diagnostics(self, source_code: str, file_path: str = "virtual_file.py") -> List[str]:
        """
        Runs static analysis on the source code to find semantic errors.
        Returns a list of human-readable (and LLM-readable) diagnostic warnings.
        """
        diagnostics =[]

        # 1. Native AST Compilation Check (Catches deep structural anomalies)
        try:
            compile(source_code, file_path, 'exec')
        except SyntaxError as e:
            return[f"SyntaxError on line {e.lineno}: {e.msg}"]

        # 2. Pyflakes Semantic Analysis (Catches undefined vars, unused imports, etc.)
        warning_stream = StringIO()
        error_stream = StringIO()

        # Pyflakes uses a reporter to handle stdout/stderr
        custom_reporter = reporter.Reporter(warning_stream, error_stream)

        # Execute analysis
        api.check(source_code, file_path, custom_reporter)

        # Parse outputs
        warnings = warning_stream.getvalue().strip().splitlines()
        errors = error_stream.getvalue().strip().splitlines()

        for w in warnings:
            # Clean up the output to make it pristine for the LLM
            clean_msg = w.replace(f"{file_path}:", "Line ")
            diagnostics.append(f"[Semantic Error] {clean_msg}")

        for e in errors:
            clean_msg = e.replace(f"{file_path}:", "Line ")
            diagnostics.append(f"[Fatal Error] {clean_msg}")

        return diagnostics

    def render_zone_2(self, source_code: str) -> str:
        """Renders the Active Diagnostics UI block."""
        diagnostics = self.get_diagnostics(source_code)

        output = []
        output.append("=== [ZONE 2: ACTIVE DIAGNOSTICS] ===")
        if not diagnostics:
            output.append("✅ No semantic or syntax errors detected.")
        else:
            for diag in diagnostics:
                output.append(f"❌ {diag}")
        output.append("====================================")
        return "\n".join(output)

# ==========================================
# DIAGNOSTIC TEST: The Semantic Taint Analysis
# ==========================================
print("Initializing Telemetry Engine...\n")
telemetry = TelemetryEngine()

# 1. Check the Genesis State (Should be clean)
print("Checking Genesis State...")
clean_source = daemon.history[0].source_code # Pull pristine code from Turn 0
print(telemetry.render_zone_2(clean_source))

# 2. Inject a Semantic Bug
print("\nInjecting Semantic Bug: Calling undefined variable `logger`...")
# We use the Graph Mutator to insert a bug right before the 'if' statement
buggy_syntax = "logger.error('This variable does not exist!')"
# Note: if_node was defined in Cell 5, we use its UUID.
success = daemon.apply_action("Insert_Before_Node", if_node.uuid, buggy_syntax)

if success:
    buggy_source = daemon.get_current_state().source_code
    print("\nChecking Turn 1 State (Bug Injected)...")
    print(telemetry.render_zone_2(buggy_source))

    # 3. Rollback the timeline to clean state
    print("\nRolling back the buggy mutation...")
    daemon.rollback()
    restored_source = daemon.get_current_state().source_code
    print(telemetry.render_zone_2(restored_source))

print("\nCell 6 Execution Complete: Telemetry Engine operational.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 6.1 MB/s eta 0:00:00
Initializing Telemetry Engine...

Checking Genesis State...
=== [ZONE 2: ACTIVE DIAGNOSTICS] ===
✅ No semantic or syntax errors detected.

Injecting Semantic Bug: Calling undefined variable `logger`...

Checking Turn 1 State (Bug Injected)...
=== [ZONE 2: ACTIVE DIAGNOSTICS] ===
❌ [Semantic Error] Line 9:13: undefined name 'logger'

Rolling back the buggy mutation...
Rolled back Turn 1. Reverted to Turn 0.
=== [ZONE 2: ACTIVE DIAGNOSTICS] ===
✅ No semantic or syntax errors detected.

Cell 6 Execution Complete: Telemetry Engine operational.


In [ ]:
# ==========================================
# CELL 7: THE AGENT ORCHESTRATOR
# ==========================================
import json
import re

SYSTEM_PROMPT = """You are an elite, Autonomous AST Engineering Agent.
You do not write plain text. You navigate and mutate code strictly via AST Graph Operations.

Your environment consists of three Zones:
[ZONE 1]: The Current File State (Semantically folded AST with UUIDs).
[ZONE 2]: Active Diagnostics (Linter errors you must fix).[ZONE 3]: Action Log (Your recent trajectory).

AVAILABLE COMMANDS (Output exactly ONE JSON block per turn):
1. Expand a folded node to see its contents:
{"action": "Expand_Node", "target_uuid": "XXXX-XXXX"}

2. Insert code BEFORE a node:
{"action": "Insert_Before_Node", "target_uuid": "XXXX-XXXX", "new_content": "your valid python code"}

3. Replace a node entirely:
{"action": "Replace_Node", "target_uuid": "XXXX-XXXX", "new_content": "your valid python code"}

INVARIANTS:
- You cannot Replace a node if it contains hidden (...) lines. Expand it first.
- Maintain exact Python indentation in your `new_content`.
"""

class AgentOrchestrator:
    def __init__(self, daemon: MCTSController, telemetry: TelemetryEngine, llm_callable):
        self.daemon = daemon
        self.telemetry = telemetry
        self.llm_callable = llm_callable # The function that calls GPT-4 / Claude / vLLM
        self.action_log = ["System initialized."]

    def generate_prompt(self) -> str:
        current_source = self.daemon.get_current_state().source_code

        zone_1 = self.daemon.render_zone_1()
        zone_2 = self.telemetry.render_zone_2(current_source)

        # Keep Zone 3 concise (last 5 actions)
        recent_logs = "\n".join(self.action_log[-5:])
        zone_3 = f"=== [ZONE 3: RECENT ACTION LOG] ===\n{recent_logs}\n==================================="

        return f"{SYSTEM_PROMPT}\n\n{zone_1}\n\n{zone_2}\n\n{zone_3}\n\nWhat is your next JSON command?"

    def _extract_json(self, text: str) -> dict:
        """Robustly extracts JSON from LLM markdown output."""
        match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
        if match:
            return json.loads(match.group(1))
        # Fallback if the LLM didn't use markdown blocks
        start = text.find('{')
        end = text.rfind('}') + 1
        if start != -1 and end != 0:
            return json.loads(text[start:end])
        raise ValueError("No valid JSON found in LLM response.")

    def step(self):
        """Executes a single turn of the agentic loop."""
        # 1. Build the prompt
        prompt = self.generate_prompt()
        print("\n" + "~"*50)
        print("🤖 [SENDING PROMPT TO LLM]:")
        print("~"*50)
        print(prompt)

        # 2. Get LLM Response
        llm_response = self.llm_callable(prompt)
        print("\n" + "~"*50)
        print("📩 [LLM RESPONSE RECEIVED]:")
        print(llm_response)
        print("~"*50)

        # 3. Parse and Execute
        try:
            command = self._extract_json(llm_response)
            action = command.get("action")
            target_uuid = command.get("target_uuid")
            new_content = command.get("new_content", "")

            if action == "Expand_Node":
                # Handle UI expansion directly
                current_ast = self.daemon.get_current_state().ast_root
                node = self.daemon.mutator._find_node(current_ast, target_uuid)
                if node:
                    node.is_folded = False
                    self.action_log.append(f"Expanded Node[UUID: {target_uuid}]")
                    print(f"✅ Successfully expanded {target_uuid}.")
                else:
                    self.action_log.append(f"Failed to expand: UUID {target_uuid} not found.")

            elif action in["Insert_Before_Node", "Insert_After_Node", "Replace_Node"]:
                success = self.daemon.apply_action(action, target_uuid, new_content)
                if success:
                    self.action_log.append(f"Successfully executed {action} on {target_uuid}")
                    print(f"✅ Mutation applied successfully.")
                else:
                    self.action_log.append(f"FAILED {action}: Sandbox rejected the mutation.")
            else:
                self.action_log.append(f"Invalid action requested: {action}")

        except Exception as e:
            error_msg = f"LLM Parsing/Execution Error: {e}"
            self.action_log.append(error_msg)
            print(f"❌ {error_msg}")

# ==========================================
# DIAGNOSTIC TEST: The Autonomous Loop
# ==========================================

# A deterministic Mock LLM to simulate intelligence
class MockLLM:
    def __init__(self, init_uuid, print_uuid):
        self.turn = 0
        self.init_uuid = init_uuid
        self.print_uuid = print_uuid

    def __call__(self, prompt: str) -> str:
        self.turn += 1
        if self.turn == 1:
            # Turn 1: The agent decides to look inside the folded __init__
            return f"""I need to see the variables initialized in the class to fix any potential state bugs.
```json
{{
  "action": "Expand_Node",
  "target_uuid": "{self.init_uuid}"
}}
```"""
        elif self.turn == 2:
            # Turn 2: The agent decides to add a new attribute to the class
            return f"""Now that I can see the state, I will add a `self.status` variable.
```json
{{
  "action": "Insert_Before_Node",
  "target_uuid": "{self.print_uuid}",
  "new_content": "self.status = 'active'"
}}
```"""
        return "I am done."

print("Initializing Agent Orchestrator with Mock LLM...\n")

# Find the UUIDs dynamically for the Mock
genesis_root = daemon.history[0].ast_root # Start from clean genesis
init_node = get_node_by_type_and_text(genesis_root, 'function_definition', 'def __init__')
print_node = get_node_by_type_and_text(genesis_root, 'expression_statement', 'print(f"Processing {item}")')

mock_llm = MockLLM(init_node.uuid, print_node.uuid)
orchestrator = AgentOrchestrator(daemon, telemetry, mock_llm)

# Run Turn 1 (Agent Expands Node)
orchestrator.step()

# Run Turn 2 (Agent Mutates Graph)
orchestrator.step()

print("\nCell 7 Execution Complete: Autonomous Loop operational.")

Initializing Agent Orchestrator with Mock LLM...


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
🤖 [SENDING PROMPT TO LLM]:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
You are an elite, Autonomous AST Engineering Agent.
You do not write plain text. You navigate and mutate code strictly via AST Graph Operations.

Your environment consists of three Zones:
[ZONE 1]: The Current File State (Semantically folded AST with UUIDs).
[ZONE 2]: Active Diagnostics (Linter errors you must fix).[ZONE 3]: Action Log (Your recent trajectory).

AVAILABLE COMMANDS (Output exactly ONE JSON block per turn):
1. Expand a folded node to see its contents:
{"action": "Expand_Node", "target_uuid": "XXXX-XXXX"}

2. Insert code BEFORE a node:
{"action": "Insert_Before_Node", "target_uuid": "XXXX-XXXX", "new_content": "your valid python code"}

3. Replace a node entirely:
{"action": "Replace_Node", "target_uuid": "XXXX-XXXX", "new_content": "your valid python code"}

INVARIANTS:
- You cannot Replace a n

In [ ]:
# ==========================================
# CELL 8: LIVE HUGGINGFACE LLM INTEGRATION
# ==========================================
!pip install -q transformers accelerate torch

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import gc

print("Downloading and loading the 7B Coder Model onto the A100 (This may take a minute)...")

# We use Qwen2.5-Coder-7B-Instruct. It is exceptionally good at strict JSON and AST logic.
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16, # Maximizes A100 tensor core efficiency
)

# Create a highly optimized generation pipeline
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,    # Low temperature for deterministic engineering tasks
    top_p=0.95,
    do_sample=True,
    return_full_text=False
)

class HuggingFaceLLM:
    """Wraps the HF pipeline to communicate with our AgentOrchestrator."""
    def __init__(self, pipe, tokenizer):
        self.pipe = pipe
        self.tokenizer = tokenizer

    def __call__(self, prompt: str) -> str:
        # We must format the prompt using the model's specific chat template
        # to activate its instruction-following weights.
        messages =[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]

        # Apply chat template
        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # Generate raw response
        outputs = self.pipe(formatted_prompt)
        response_text = outputs[0]["generated_text"].strip()
        return response_text

print("\nModel loaded successfully! Initializing Live Orchestrator...")

# 1. Initialize the live AI
live_llm = HuggingFaceLLM(hf_pipeline, tokenizer)
live_orchestrator = AgentOrchestrator(daemon, telemetry, live_llm)

# 2. Reset the MCTS Daemon to the Genesis state
while len(daemon.history) > 1:
    daemon.rollback()
print("Timeline reverted to Genesis State.")

# 3. Inject a Bug Intentionally to Test the AI
print("\n[SYSTEM]: Injecting the 'logger' bug to see if the AI can fix it...")
genesis_root = daemon.get_current_state().ast_root
if_node = get_node_by_type_and_text(genesis_root, 'if_statement', 'if item < 0:')

if if_node:
    # We forcefully insert the bug
    daemon.apply_action("Insert_Before_Node", if_node.uuid, "logger.error('Negative value!')")
    print("\n[SYSTEM]: Bug injected successfully. Handing control to the LLM.")

    # Let the LLM take a turn to fix it!
    # Zone 2 will currently show: `[Semantic Error] Line X: undefined name 'logger'`
    live_orchestrator.step()

    # We can execute a second step to see its follow-up action
    print("\n[SYSTEM]: Allowing LLM to take a second turn if needed...")
    live_orchestrator.step()
else:
    print("Test setup failed: Could not find the target node.")

print("\nCell 8 Execution Complete: Live AI integration test finished.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model loaded successfully! Initializing Live Orchestrator...
Rolled back Turn 1. Reverted to Turn 0.
Timeline reverted to Genesis State.

[SYSTEM]: Injecting the 'logger' bug to see if the AI can fix it...

[SYSTEM]: Bug injected successfully. Handing control to the LLM.

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
🤖 [SENDING PROMPT TO LLM]:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
You are an elite, Autonomous AST Engineering Agent.
You do not write plain text. You navigate and mutate code strictly via AST Graph Operations.

Your environment consists of three Zones:
[ZONE 1]: The Current File State (Semantically folded AST with UUIDs).
[ZONE 2]: Active Diagnostics (Linter errors you must fix).[ZONE 3]: Action Log (Your recent trajectory).

AVAILABLE COMMANDS (Output exactly ONE JSON block per turn):
1. Expand a folded node to see its contents:
{"action": "Expand_Node", "target_uuid": "XXXX-XXXX"}

2. Insert code BEFORE a node:
{"action": "Insert_Before_Node", "target_

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
📩 [LLM RESPONSE RECEIVED]:
```json
{
  "action": "Expand_Node",
  "target_uuid": "6127-E64F"
}
```
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
✅ Successfully expanded 6127-E64F.

[SYSTEM]: Allowing LLM to take a second turn if needed...

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
🤖 [SENDING PROMPT TO LLM]:
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
You are an elite, Autonomous AST Engineering Agent.
You do not write plain text. You navigate and mutate code strictly via AST Graph Operations.

Your environment consists of three Zones:
[ZONE 1]: The Current File State (Semantically folded AST with UUIDs).
[ZONE 2]: Active Diagnostics (Linter errors you must fix).[ZONE 3]: Action Log (Your recent trajectory).

AVAILABLE COMMANDS (Output exactly ONE JSON block per turn):
1. Expand a folded node to see its contents:
{"action": "Expand_Node", "target_uuid": "XXXX-XXXX"}

2. Insert code BEFORE a node:
{"action": "Insert